# 7.1 Neural Networks Framing and Use Cases

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook introduces when neural networks help, where they are overkill, and how their workflow differs from the statistics and classical machine-learning blocks. The final subsection gives one TensorFlow / Keras orientation example, then the course standardizes on PyTorch.


## 7.1.1 What neural nets solve well

### Why It Matters
Neural networks become useful when the relationship between inputs and outputs is flexible, layered, or highly nonlinear. A small XOR example shows why a linear boundary can fail while a tiny multilayer network can succeed.


In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

linear = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
mlp = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1), nn.Sigmoid())

def train(model, steps=2000, lr=0.1):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    for _ in range(steps):
        optimizer.zero_grad()
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        acc = ((model(X) > 0.5).float() == y).float().mean().item()
    return round(acc, 3)

{
    "linear_accuracy": train(linear),
    "mlp_accuracy": train(mlp),
}


### Interpretation
The XOR pattern is the standard reminder that linear structure is not enough for every problem. Even a very small hidden layer can create the curved decision boundary that a single linear unit cannot.


## 7.1.2 Function approximation intuition

### Why It Matters
A neural network is a parameterized function approximator. Here a small multilayer perceptron learns a noisy sine curve and shows that the model is fitting a shape rather than memorizing a formula.


In [ ]:
import torch
from torch import nn

torch.manual_seed(8)
x = torch.linspace(-3.14, 3.14, 120).unsqueeze(1)
y = torch.sin(x) + 0.08 * torch.randn_like(x)

model = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()

for _ in range(500):
    optimizer.zero_grad()
    preds = model(x)
    loss = loss_fn(preds, y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    fitted = model(x)
    mse = loss_fn(fitted, y).item()
{
    "final_mse": round(mse, 4),
    "sample_predictions": fitted[:5, 0].round(decimals=3).tolist(),
}


### Interpretation
The network is not told anything about sine waves. It learns a set of weights that maps inputs to outputs well enough to trace the pattern in the data.


## 7.1.3 Tabular versus image versus sequence tasks

### Why It Matters
The same training ideas reappear across modalities, but the input shapes and model components change. This example contrasts tensor layouts for tabular, image, and sequence data.


In [ ]:
import torch
from torch import nn

tabular = torch.randn(16, 12)
images = torch.randn(16, 1, 8, 8)
sequences = torch.randint(0, 50, (16, 6))

tabular_head = nn.Linear(12, 3)
image_head = nn.Sequential(nn.Conv2d(1, 4, kernel_size=3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4 * 8 * 8, 3))
sequence_head = nn.Sequential(nn.Embedding(50, 8), nn.Flatten(), nn.Linear(6 * 8, 3))

{
    "tabular_output_shape": list(tabular_head(tabular).shape),
    "image_output_shape": list(image_head(images).shape),
    "sequence_output_shape": list(sequence_head(sequences).shape),
}


### Interpretation
The learning objective can still be classification in all three cases, but the representation and architecture must respect the structure of the data.


## 7.1.4 When simpler models are still preferable

### Why It Matters
A neural network is not automatically the best first choice. On a nearly linear tabular problem, a simple logistic model can match a small MLP without extra tuning or compute.


In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=320, n_features=6, n_informative=5, n_redundant=0, class_sep=1.8, random_state=9)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=9)

lr = LogisticRegression(max_iter=2000).fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))

Xt = torch.tensor(X_train, dtype=torch.float32)
yt = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
Xv = torch.tensor(X_test, dtype=torch.float32)

mlp = nn.Sequential(nn.Linear(6, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
opt = torch.optim.Adam(mlp.parameters(), lr=0.02)
loss_fn = nn.BCELoss()
for _ in range(250):
    opt.zero_grad()
    loss = loss_fn(mlp(Xt), yt)
    loss.backward()
    opt.step()
with torch.no_grad():
    mlp_preds = (mlp(Xv) > 0.5).int().numpy().ravel()
{
    "logistic_accuracy": round(float(lr_acc), 3),
    "mlp_accuracy": round(float((mlp_preds == y_test).mean()), 3),
}


### Interpretation
On clean, low-dimensional tabular data, the neural net may not buy much. A baseline-first mindset still matters in the neural-networks chapter.


## 7.1.5 Capacity, data needs, and compute tradeoffs

### Why It Matters
Wider and deeper networks carry more parameters. That increases flexibility, but it also increases optimization cost and the amount of data you usually need for stable generalization.


In [ ]:
import torch
from torch import nn

small = nn.Sequential(nn.Linear(20, 8), nn.ReLU(), nn.Linear(8, 1))
large = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

def count_params(model):
    return sum(p.numel() for p in model.parameters())

X = torch.randn(512, 20)
y = torch.randn(512, 1)

def one_epoch_time_proxy(model):
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
    return round(float(loss.item()), 4)

{
    "small_params": count_params(small),
    "large_params": count_params(large),
    "small_one_step_loss": one_epoch_time_proxy(small),
    "large_one_step_loss": one_epoch_time_proxy(large),
}


### Interpretation
Parameter count is not a quality metric by itself, but it is a useful way to reason about model capacity, memory footprint, and optimization burden.


## 7.1.6 One TensorFlow / Keras orientation example, then PyTorch

### Why It Matters
This course uses PyTorch everywhere after this point. The cell below shows a single TensorFlow / Keras reference implementation when available, and always shows the equivalent PyTorch workflow that the rest of the notebooks will follow.


In [ ]:
import importlib.util
import textwrap
import torch
from torch import nn

keras_reference = textwrap.dedent('''
import tensorflow as tf

keras_model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
keras_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
keras_model.fit(X_np, y_np, epochs=20, verbose=0)
''').strip()

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], dtype=torch.float32)
y = torch.tensor([[0.], [1.], [1.], [0.]], dtype=torch.float32)
torch_model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
opt = torch.optim.Adam(torch_model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()
for _ in range(300):
    opt.zero_grad()
    loss = loss_fn(torch_model(X), y)
    loss.backward()
    opt.step()

result = {
    "tensorflow_available": bool(importlib.util.find_spec("tensorflow")),
    "pytorch_accuracy": round(float((((torch_model(X) > 0.5).float() == y).float().mean()).item()), 3),
    "keras_reference_snippet": keras_reference,
}
result


### Interpretation
The pedagogical point is API orientation, not framework comparison. After this subsection, the course uses PyTorch only.
